# Regular Expressions

The `re` module gives Python a full regular-expression engine. Patterns let you search, extract, validate, and transform text far more expressively than string methods alone, and compiled patterns can be reused efficiently across large inputs.

**What's inside:** `search`, `match`, `fullmatch`, `findall`, `finditer`, `sub`, `split`, capture groups, named groups, flags, lookahead/lookbehind, and `re.VERBOSE` for readable patterns.

**Learn more:** [re: Regular expression operations](https://docs.python.org/3/library/re.html) · [regex HOWTO](https://docs.python.org/3/howto/regex.html)

## 1. Basic matching

### 1.1 search: find anywhere in the string

In [1]:
import re

text = 'The price is $42.99'

m = re.search(r'\d+\.\d+', text)
if m:
    print(m.group())       # '42.99'
    print(m.start(), m.end())   # position in string

42.99
14 19


### 1.2 match vs fullmatch

In [2]:
import re

# match: anchored at the START of the string
print(re.match(r'\d+', '123abc'))   # matches '123'
print(re.match(r'\d+', 'abc123'))   # None: doesn't start with digits

# fullmatch: the ENTIRE string must match
print(re.fullmatch(r'\d{4}-\d{2}-\d{2}', '2024-03-15'))   # matches
print(re.fullmatch(r'\d{4}-\d{2}-\d{2}', '2024-03-15x'))  # None

<re.Match object; span=(0, 3), match='123'>
None
<re.Match object; span=(0, 10), match='2024-03-15'>
None


## 2. findall and finditer

In [3]:
import re

text = 'Call 555-1234 or 555-5678 for info.'

# findall returns a list of all matches
phones = re.findall(r'\d{3}-\d{4}', text)
print(phones)   # ['555-1234', '555-5678']

# finditer returns an iterator of match objects (memory-efficient)
for m in re.finditer(r'\d{3}-\d{4}', text):
    print(m.group(), 'at', m.start())

['555-1234', '555-5678']
555-1234 at 5
555-5678 at 17


## 3. Capture groups

### 3.1 Positional groups

In [4]:
import re

log_line = '2024-03-15 14:32:01 ERROR connection refused'

m = re.search(r'(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2}) (\w+)', log_line)
if m:
    print(m.group(0))   # full match
    print(m.group(1))   # '2024-03-15'
    print(m.group(2))   # '14:32:01'
    print(m.group(3))   # 'ERROR'
    print(m.groups())   # ('2024-03-15', '14:32:01', 'ERROR')

2024-03-15 14:32:01 ERROR
2024-03-15
14:32:01
ERROR
('2024-03-15', '14:32:01', 'ERROR')


### 3.2 Named groups: `(?P<name>...)`

In [5]:
import re

pattern = r'(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+)'
log_line = '2024-03-15 14:32:01 ERROR connection refused'

m = re.search(pattern, log_line)
if m:
    print(m.group('date'))    # '2024-03-15'
    print(m.group('level'))   # 'ERROR'
    print(m.groupdict())      # {'date': ..., 'time': ..., 'level': ...}

2024-03-15
ERROR
{'date': '2024-03-15', 'time': '14:32:01', 'level': 'ERROR'}


### 3.3 findall with groups

In [6]:
import re

text = 'Alice:30, Bob:25, Carol:28'

# with one group: list of group values
ages = re.findall(r'\w+:(\d+)', text)
print(ages)   # ['30', '25', '28']

# with multiple groups: list of tuples
pairs = re.findall(r'(\w+):(\d+)', text)
print(pairs)  # [('Alice', '30'), ('Bob', '25'), ('Carol', '28')]

['30', '25', '28']
[('Alice', '30'), ('Bob', '25'), ('Carol', '28')]


## 4. sub: search and replace

In [7]:
import re

# basic substitution
text = 'Hello   world   !'
clean = re.sub(r'\s+', ' ', text)
print(clean)   # 'Hello world !'

# back-references in replacement string
dates = 'Today is 15/03/2024, tomorrow is 16/03/2024'
iso = re.sub(r'(\d{2})/(\d{2})/(\d{4})', r'\3-\2-\1', dates)
print(iso)     # 'Today is 2024-03-15, tomorrow is 2024-03-16'

Hello world !
Today is 2024-03-15, tomorrow is 2024-03-16


In [8]:
import re

# function as replacement, called for each match
def censor(m):
    word = m.group()
    return word[0] + '*' * (len(word) - 2) + word[-1]

text = 'Password is secret and token is hidden'
print(re.sub(r'\b\w{4,}\b', censor, text))

P******d is s****t and t***n is h****n


## 5. split

In [9]:
import re

# split on any whitespace sequence
print(re.split(r'\s+', 'one   two\tthree\nfour'))

# split on punctuation; keep delimiters using a capture group
sentence = 'Hello! How are you? Fine, thanks.'
print(re.split(r'([!?.,])', sentence))

['one', 'two', 'three', 'four']
['Hello', '!', ' How are you', '?', ' Fine', ',', ' thanks', '.', '']


## 6. Flags

In [10]:
import re

text = 'Python is AWESOME\npython rocks'

# re.IGNORECASE (re.I)
print(re.findall(r'python', text, re.IGNORECASE))   # ['Python', 'python']

# re.MULTILINE (re.M): ^ and $ match at line boundaries
print(re.findall(r'^python', text, re.IGNORECASE | re.MULTILINE))

# re.DOTALL (re.S): dot matches newline too
print(re.search(r'AWESOME.python', text, re.DOTALL).group())

['Python', 'python']
['Python', 'python']
AWESOME
python


## 7. re.compile: reuse patterns

In [11]:
import re

# compile once, use many times; faster for repeated matching
IPV4 = re.compile(
    r'(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}'
    r'(?:25[0-5]|2[0-4]\d|[01]?\d\d?)'
)

lines = [
    'Server at 192.168.1.1 is up',
    'Gateway 10.0.0.1 unreachable',
    'No address here',
]
for line in lines:
    m = IPV4.search(line)
    if m:
        print(m.group())

192.168.1.1
10.0.0.1


## 8. Lookahead and lookbehind

In [12]:
import re

prices = 'apple $1.20, banana $0.50, cherry $2.99'

# positive lookahead (?=...): match digits before a specific context
nums_before_dollar = re.findall(r'\$([\d.]+)', prices)
print(nums_before_dollar)   # ['1.20', '0.50', '2.99']

# positive lookbehind (?<=...): match only what follows a pattern
words_after_colon = re.findall(r'(?<=: )\w+', 'Name: Alice, City: Paris')
print(words_after_colon)    # ['Alice', 'Paris']

# negative lookahead (?!...): words not followed by a digit
words = re.findall(r'\b\w+\b(?!\d)', 'foo1 bar baz2 qux')
print(words)

['1.20', '0.50', '2.99']
['Alice', 'Paris']
['foo1', 'bar', 'baz2', 'qux']


## 9. re.VERBOSE: readable patterns

In [13]:
import re

# re.VERBOSE (re.X) allows whitespace and comments inside the pattern
EMAIL = re.compile(r"""
    (?P<user>[a-zA-Z0-9._%+\-]+)   # local part
    @                               # literal @
    (?P<domain>[a-zA-Z0-9.\-]+)   # domain
    \.                              # literal dot
    (?P<tld>[a-zA-Z]{2,})          # top-level domain
""", re.VERBOSE)

for addr in ['alice@example.com', 'bad@', 'bob.smith@mail.co.uk']:
    m = EMAIL.fullmatch(addr)
    print(f'{addr!r:30} → {m.groupdict() if m else "no match"}')

'alice@example.com'            → {'user': 'alice', 'domain': 'example', 'tld': 'com'}
'bad@'                         → no match
'bob.smith@mail.co.uk'         → {'user': 'bob.smith', 'domain': 'mail.co', 'tld': 'uk'}
